In [1]:
import numpy as np
from dask.distributed import Client

from tensor_regressor.synthetic_data import generate_low_rank_data
from tensor_regressor.multilinear_ops import mode_n_product, multi_mode_product
from tensor_regressor.models.tensor_decomp.tucker.hosvd import HoSVD
from tensor_regressor.models.tensor_decomp.tucker.hooi import HoOI

In [2]:
client = Client(n_workers=4)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 192,Total memory: 755.36 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37699,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:38233,Total threads: 48
Dashboard: http://127.0.0.1:37089/status,Memory: 188.84 GiB
Nanny: tcp://127.0.0.1:41471,


In [3]:
%%timeit -n 1 -r 1
futures = []
X = generate_low_rank_data((400,400,400), (5,6,7))
h = HoSVD(X)
C, Us = h()

1.01 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [8]:
%%timeit -n 1 -r 1
futures = []
X = generate_low_rank_data((500,500,500), (5,6,7), seed=10)
# h = HoSVD(X, client=client)
h = HoSVD(X, core_dims=(5,6,7))
C, Us = h()
print(np.isclose(X, multi_mode_product(C, Us, [1,2,3]).cpu().numpy()).all())

True
2.49 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [9]:
%%timeit -n 1 -r 1
futures = []
X = generate_low_rank_data((500,500,500), (5,6,7))
h = HoSVD(X, core_dims=(5,6,7))
# h = HoSVD(X)
C, Us = h()

881 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [11]:
X = generate_low_rank_data((200,200,200), (5,6,7), seed=10)
hooi = HoOI(X, (7,7,7), max_it=40)
C, Us = hooi()
print(np.isclose(X, multi_mode_product(C, Us, [1,2,3]).cpu().numpy()).all())

Iteration 0: ||X|| - ||C|| = 2.972910806420259e-11
True


In [12]:
client.shutdown()